In [16]:
import yfinance as yf
import pandas as pd
import numpy as np

sp500 = yf.download('^GSPC', start='1950-01-01', end='2026-08-21')
close = sp500['Close'].dropna()

if isinstance(close, pd.DataFrame):
    close = close.iloc[:, 0]

close = close.sort_index()

running_max = close.cummax()

is_new_ath = close >= running_max
regime_id = is_new_ath.cumsum()

df = pd.DataFrame({
    'close': close,
    'running_max': running_max,
    'regime_id': regime_id
})

regime_groups = df.groupby('regime_id')

results = []
for rid, group in regime_groups:
    ath_price = group['running_max'].iloc[0] 
    ath_date = group.index[0]
    
    trough_price = group['close'].min()
    trough_date = group['close'].idxmin()
    
    after_trough = group[group.index >= trough_date]
    recovery_date = after_trough[after_trough['close'] >= ath_price].index.min()
    
    drawdown_pct = (ath_price - trough_price) / ath_price * 100
    
    results.append({
        'ath_date': ath_date,
        'ath_price': ath_price,
        'trough_date': trough_date,
        'trough_price': trough_price,
        'drawdown_pct': drawdown_pct,
        'recovery_date': recovery_date,
        'duration_to_trough_days': (trough_date - ath_date).days,
        'duration_to_recovery_days': (recovery_date - ath_date).days if pd.notna(recovery_date) else np.nan
    })

corrections_df = pd.DataFrame(results)


significant = corrections_df[corrections_df['drawdown_pct'] >= 5].copy()

print(f"Number of significant corrections: {len(significant)}")
print(significant.sort_values('drawdown_pct', ascending=False).head(10))

[*********************100%***********************]  1 of 1 completed


Number of significant corrections: 74
       ath_date    ath_price trough_date  trough_price  drawdown_pct  \
1067 2007-10-09  1565.150024  2009-03-09    676.530029     56.775388   
1058 2000-03-24  1527.459961  2002-10-09    776.760010     49.146948   
554  1973-01-11   120.239998  1974-10-03     62.279999     48.203593   
519  1968-11-29   108.370003  1970-05-26     69.290001     36.061641   
1322 2020-02-19  3386.149902  2020-03-23   2237.399902     33.924960   
731  1987-08-25   336.769989  1987-12-04    223.919998     33.509515   
348  1961-12-12    72.639999  1962-06-26     52.320000     27.973568   
578  1980-11-28   140.520004  1982-08-12    102.419998     27.113582   
1413 2022-01-03  4796.560059  2022-10-12   3577.030029     25.425097   
471  1966-02-09    94.059998  1966-10-07     73.199997     22.177335   

     recovery_date  duration_to_trough_days  duration_to_recovery_days  
1067           NaT                      517                        NaN  
1058           NaT     

In [17]:

print("\nUsing duration to TROUGH:")
print(significant['duration_to_trough_days'].quantile([0.25, 0.5, 0.75]))

print("\nUsing duration to RECOVERY:")
print(significant['duration_to_recovery_days'].quantile([0.25, 0.5, 0.75]))

print("\nDrawdown percentiles (%):")
print(significant['drawdown_pct'].quantile([0.25, 0.5, 0.75]))

median_drawdown = significant['drawdown_pct'].median()
print(f"\nMedian drawdown: {median_drawdown:.2f}%")


Using duration to TROUGH:
0.25    22.00
0.50    40.50
0.75    86.25
Name: duration_to_trough_days, dtype: float64

Using duration to RECOVERY:
0.25   NaN
0.50   NaN
0.75   NaN
Name: duration_to_recovery_days, dtype: float64

Drawdown percentiles (%):
0.25     6.234677
0.50     7.986358
0.75    14.019826
Name: drawdown_pct, dtype: float64

Median drawdown: 7.99%
